In [17]:
import torch
from torch import nn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os
import random

In [18]:
FILE_PATH = "Topic3_healthcare_analytics_dataset.csv"
RANDOM_SEED = 23
DEVICE = "cuda" if torch.cuda.is_available() else "cpu" #if a gpu is available -> take it instead of CPU
TRAINING_BATCH_SIZE = 128
TEST_BATCH_SIZE = 256

In [19]:
def set_seed(seed: int): #for reproduciblity
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)      # for GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [20]:
class Model(nn.Module):
    def __init__(self, in_features: int, hidden_units: int, out_features):
        super().__init__()
        #create a sequential Model layout
        self.Layer_stack = nn.Sequential(
            nn.Linear(in_features=in_features, out_features=hidden_units),
            nn.ReLU(), #activation function

            nn.Linear(in_features=hidden_units, out_features=hidden_units),
            nn.ReLU(),
            nn.Dropout(0.1), #dropout layer

            nn.Linear(in_features=hidden_units, out_features=hidden_units),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(in_features=hidden_units, out_features=hidden_units),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(in_features=hidden_units, out_features=hidden_units),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(in_features=hidden_units, out_features=out_features),
        )

    def forward(self, X):
        return self.Layer_stack(X) #takes data and fits it trough the layers

In [21]:
class Preprocessor:
    def __init__(self):
        self.categories = {} #holds a list of unique elements per column example: {'A': [1,2,...], 'B':["x", "y",...], ...}
        self.label_mapping = {} #holds the label groups mapped to their int mapping example: {'0-10': 0, '11-20': 1, ...}
        self.feature_columns = [] #saves list with all feature columns names (helps to allign the data in preiction) example: ['Ward_Type_P', 'Ward_Type_Q', 'Ward_Type_R'...]

        self.HOT_CODE_COLUMNS = [
            "Hospital_code", "Hospital_type_code", "City_Code_Hospital",
            "Hospital_region_code", "Department", "Ward_Type",
            "Ward_Facility_Code", "Type of Admission", "City_Code_Patient",
            "Age", "Bed Grade", "Severity of Illness"
        ]
        self.NORMALIZE_COLUMNS = [
            "Admission_Deposit", "Visitors with Patient",
            "Available Extra Rooms in Hospital",  # only the truly numeric ones
        ]
        self.LABEL_COLUMN = "Stay"
        self.DROP_COLUMNS = ["case_id", "patientid"]

        self.means = {} #holds mean of each NORMALIZE_COLUMS ecample: {'Admission_Deposit': np.float64(4.), 'Visitors with Patient': np.float64(3.), ...}
        self.stds = {} #holds list of standard deviations per NORMALIZE COLUMNS structure same as means

    def save(self, path="preprocessor.pt"):
        '''
        saves the stats of the Preprocessor.
        - categories (list of unique entries per column)
        - means (mean for each NORMALIZE COLUMN (only over training data))
        - stds (standard deviation for each NORMALIZE_COLUMNS (only over training data))
        - label_mapping (list each stay group mapped to an int)
        - feature_columns (list of names of all features for the model
        '''
        torch.save({
            "categories": self.categories,
            "means": self.means,
            "stds": self.stds,
            "label_mapping": self.label_mapping,
            "feature_columns": self.feature_columns,
        }, path)

    def load(self, path="preprocessor.pt"):
        '''loads the saved Preprocessor stats from training'''
        state = torch.load(path, weights_only=False)
        self.categories = state["categories"]
        self.means = state["means"]
        self.stds = state["stds"]
        self.label_mapping = state["label_mapping"]
        self.feature_columns = state["feature_columns"]

    def create_hot_code_mapping(self, data: pd.DataFrame):
        '''it fills self.categories with a list of unique entries per column'''
        for column in self.HOT_CODE_COLUMNS:
            self.categories[column] = sorted(data[column].dropna().unique().tolist())

    def apply_mapping(self, data: pd.DataFrame)->pd.DataFrame:
        '''takes a DataFrame and for each column in HOT_CODE_COLUMNS it splits it into new columns, 1 column per unique entries. Also it normalizes the columns of the original DataFrame that are stated in NORMALIZE_COLUMNS. It returns a new data Frame with structure: [Normalize_columns | hot coded columns]'''
        new_columns = {}
        for value in self.categories.items(): #{'A': [1,2,...], ...} takes the item ('A', [])
            for unique_value in value[1]: # takes the value []
                column_name = f"{value[0]}_{unique_value}" #creates a name for the new column
                new_columns[column_name] = (data[value[0]] == unique_value).astype(int) #row that = unique value -> 1 rest = 0

        hot = pd.DataFrame(new_columns, index=data.index)  #creates new DataFrame with the code coded columns in it

        # carry over the numeric columns that aren't hot-coded (from original DataFrame
        passthrough = data[self.NORMALIZE_COLUMNS].copy()

        full = pd.concat([passthrough, hot], axis=1) #merge DataFrames
        if self.means:  # only normalize once stats are learned
            full = self.normalize_data(full) #normalize the columns that come from the original DataFrame
        return full

    def map_label_columns_to_int(self, data:pd.DataFrame):
        '''takes the label column and maps each unique label to an int'''
        mapping = {}
        unique_data_points = sorted(data[self.LABEL_COLUMN].unique())  # get the unique data points per column in order
        mapping_values = [i for i in range(len(unique_data_points))]  # create a number (int) for each unique datapoint

        for data_point, mapping_value in zip(unique_data_points, mapping_values):  # iterates through both lists at the same time
            mapping[data_point] = mapping_value  # map a int value to the group and append in dict

        self.label_mapping = mapping #saves the mapping
        return mapping

    def normalize_data(self, data:pd.DataFrame)->pd.DataFrame:
        '''normalizes columns in NORMALIZE_COLUMNS'''
        data = data.copy()
        for column in self.NORMALIZE_COLUMNS:
            data[column] = (data[column] - self.means[column]) / self.stds[column] #every entry in every column (only NORMALIZE_COLUMNS
        return data

    def get_test_train_split(self, data: pd.DataFrame):
        '''splits the Data into train and test Data, hot-codes, normalize selected data and maps labels to an int. It returns 4 Tensors'''
        print("start loading train_test split...")

        # 1. split RAW data first (stats must come from raw train only)
        train_data, test_data = train_test_split(
            data, test_size=0.2, random_state=RANDOM_SEED, stratify=data[self.LABEL_COLUMN]
        )

        # 2. learn normalization stats from raw training rows
        for column in self.NORMALIZE_COLUMNS: #only NORMALIZE_COLUMNS
            self.means[column] = train_data[column].mean() #get mean for selected col (only with respect to training data) and save it
            self.stds[column] = train_data[column].std() #get Standard deviation selected col (only with respect to training data) and save it

        # 3. transform train and test — apply_mapping now normalizes (self.means is set), ONCE each
        X_train = self.apply_mapping(train_data)
        X_test = self.apply_mapping(test_data)
        self.feature_columns = X_train.columns.tolist()

        # 4. Transform labels labels for train and test
        mapping = self.map_label_columns_to_int(data)
        y_train = train_data[self.LABEL_COLUMN].map(mapping)
        y_test = test_data[self.LABEL_COLUMN].map(mapping)

        # 5. transform the data into tensors
        X_train = torch.tensor(X_train.values, dtype=torch.float32, device=DEVICE)
        X_test = torch.tensor(X_test.values, dtype=torch.float32, device=DEVICE)
        y_train = torch.tensor(y_train.values, dtype=torch.int64, device=DEVICE)
        y_test = torch.tensor(y_test.values, dtype=torch.int64, device=DEVICE)

        print("Done loading train_test_split")
        return X_train, X_test, y_train, y_test

    def predict_row(self, row_index, data, model, print_col=True):
        row = data.iloc[[row_index]] #extract the given row from the given DataFrame

        if print_col: #if needed print the values of each column of the row
            print(row)

        X = self.apply_mapping(row) #formats to usable input data for the model
        X = X.reindex(columns=self.feature_columns, fill_value=0) #bring the columns of the dataframe into the correct order (the order that is given by the training)
        X = torch.tensor(X.values, dtype=torch.float32, device=DEVICE) #transforms the formated data into a tensor

        with torch.inference_mode(): #turn of gradient tracking
            logits = model(X) #get the models raw output scores of the model
            idx = logits.argmax(dim=1).item() #select the output class with the highest score -> this is the prediction class

        inverse = {code: value for value, code in self.label_mapping.items()} #reverse keys and values of the label_mapping {"0-10": 0,...} -> {0: "0-10",...}
        pred = inverse[idx] #get the class coresponding to the index

        return pred




In [22]:
#accuracy function for the model
def accuracy_fn(y_pred, y_true):
    correct = torch.eq(y_true, y_pred).sum().item()  # compares how many y_true are equal to y_pred
    acc = (correct / len(y_pred)) * 100
    return acc

In [23]:
data = pd.read_csv(FILE_PATH) #load csv data
pre = Preprocessor()

In [24]:
#training and saving a model for prediction
set_seed(RANDOM_SEED)

if not os.path.exists("model_final.pt"): #only train if not pre-trained model exists
    pre.create_hot_code_mapping(data)
    X_train, X_test, y_train, y_test = pre.get_test_train_split(data) #split data and transform them into tensors
    out_features = len(pre.label_mapping)
    model = Model(in_features=len(pre.feature_columns), hidden_units=512, out_features=out_features).to(DEVICE) #create a Model

    #choose optimizer and loss function
    optimizer = torch.optim.Adam(params=model.parameters(), lr=0.001)
    loss_fn = nn.CrossEntropyLoss()

    # training loop
    epoch = 0
    break_counter = 0 #tracks how often in a row the new test_loss is not smaller then the previouse best loss
    best_loss = float("inf")
    while True:
        epoch += 1
        # get a random Batch
        permutation = torch.randperm(len(X_train))  # get random index list for random batches
        train_loss, train_acc = 0, 0 #reset train_loss and train_accuracy for new epoch
        batch_count = 0 #track in which batch we are in

        model.train()
        for start in range(0, len(X_train), TRAINING_BATCH_SIZE):
            batch_count += 1
            indexes = permutation[start:start + TRAINING_BATCH_SIZE] #get indexes from premutation (it basacilly shuffles the data
            X_batch, y_batch = X_train[indexes], y_train[indexes] #get a Batch

            y_preds = model(X_batch) #intput batch into model and get back the class scores
            loss = loss_fn(y_preds, y_batch) #calculate the loss for this batch
            acc = accuracy_fn(y_preds.argmax(dim=1), y_batch) #accuracy score of this batch

            #add loss and accuracy for the oss and acc over the entire data set
            train_loss += loss.item()
            train_acc += acc

            optimizer.zero_grad() #set gradient to zero such that it doesn't accumulate over time
            loss.backward() #calculate the gradient for this loss (what impact does each parameter has on the loss function)
            optimizer.step() #adjust the parameters of the model based on the calculated gradient

        train_loss /= batch_count #get overall loss of the model over the entire data set
        train_acc /= batch_count #get overall accuracy of the model over the entire data set

        with torch.inference_mode(): #turn of gradient tracking for evaluation of the model
            test_loss, test_acc = 0, 0
            batch_count = 0

            model.eval()
            #no "shuffling" needed for evaluation
            for start in range(0, len(X_test), TEST_BATCH_SIZE):
                #get a batch
                X_batch_test = X_test[start:start + TEST_BATCH_SIZE]
                y_batch_test = y_test[start:start + TEST_BATCH_SIZE]

                batch_count += 1

                y_preds_test = model(X_batch_test) #get prediction of the model on the batch
                loss = loss_fn(y_preds_test, y_batch_test) #calculate loss of the batch
                acc = accuracy_fn(y_preds_test.argmax(dim=1), y_batch_test) #calculate accuracy of the batch

                #add loss and acc
                test_loss += loss.item()
                test_acc += acc

            #calculate loss and accuracy of the model over the entire Data set
            test_loss /= batch_count
            test_acc /= batch_count

            print(f"{epoch} | train_loss: {train_loss:.5f}, test_loss: {test_loss:.5f}, train_acc: {train_acc:.3f} test_acc: {test_acc:.3f} counter: {break_counter}")

            if test_loss <= best_loss: #check if the current test_loss is better (lower) then the current best loss saved
                best_loss = test_loss #if the loss is lower, then save it as the new best loss
                break_counter = 0
                torch.save(model.state_dict(), "checkpoint.pth") #save the best current model parameters

            else: #if the current test_loss is not lower then the current best loss -> increase the counter
                break_counter += 1

            #if the model didn't achieve a better test loss (lower) then the best loss in 10 epochs, then end training
            if break_counter >= 10:
                break

    pre.save() #save the states

    #rebuild the best model, and load best weights
    export_model = Model(
        in_features=len(pre.feature_columns),
        hidden_units=512,
        out_features=out_features,
    )

    export_model.load_state_dict(torch.load("checkpoint.pth", weights_only=True)) #load the parameters of the lowest test_loss epoch into the model
    export_model.eval()
    torch.jit.script(export_model).save("model_final.pt")   #save the model as a self_contained model
    print("saved preprocessor.pt + model_final.pt")

else:
    print("found pre-trained model: model_final.pt — skipping training")


found pre-trained model: model_final.pt — skipping training


In [25]:
# load preprocessor and evaluation model
pre = Preprocessor()
pre.load("preprocessor.pt")
model = torch.jit.load("model_final.pt", map_location=DEVICE) #load the model for evaluation
model.eval() #sets model into evaluation mode

# get the SAME test split (seed is fixed, so reproducible)
set_seed(RANDOM_SEED)
_, X_test, _, y_test = pre.get_test_train_split(data)

with torch.inference_mode():
    logits = model(X_test) #get class output scores
    total_acc = accuracy_fn(logits.argmax(dim=1), y_test) #calculate the accuracy over the entire test data set

print(f"Test accuracy: {total_acc:.2f}%")

start loading train_test split...
Done loading train_test_split
Test accuracy: 42.32%


In [ ]:
# predict on set amount of random rows (num_test_rows) and compare to the true Stay
num_test_rows = 30 # set the amount of rows to test model on
random_indices = random.sample(range(len(data)), num_test_rows)  #  num_test_rows distinct row positions (change variable above if necessary)

correct = 0
for row_index in random_indices: #interates through the random_indices list
    pred = pre.predict_row(row_index, data=data, model=model, print_col=False) #get the prediction of the model on this row
    true_val = data.iloc[row_index][pre.LABEL_COLUMN] #extract the true Stay from the DataFrame
    print(f"pred: {pred} | true: {true_val}")
    if pred == true_val:
        correct += 1

print(f"\n{correct}/{num_test_rows} correct")
correct_percentage = round(correct / num_test_rows * 100, 2)
print(f"(This equals roughly {correct_percentage}% accuracy)")
#also includes rows from training it is just as a demo

{'Admission_Deposit': np.float64(4882.761012452496), 'Visitors with Patient': np.float64(3.279711114829067), 'Available Extra Rooms in Hospital': np.float64(3.1971087581761832)}
pred: 51-60 | true: 31-40
pred: 21-30 | true: 0-10
pred: 11-20 | true: 11-20
pred: 21-30 | true: 21-30
pred: 11-20 | true: 0-10
pred: 0-10 | true: 11-20
pred: 31-40 | true: 31-40
pred: 11-20 | true: 31-40
pred: 11-20 | true: 0-10
pred: 21-30 | true: 41-50
pred: 21-30 | true: 21-30
pred: 21-30 | true: 21-30
pred: 21-30 | true: 21-30
pred: 11-20 | true: 11-20
pred: 11-20 | true: 11-20
pred: 51-60 | true: 71-80
pred: 21-30 | true: 11-20
pred: 0-10 | true: 51-60
pred: 21-30 | true: 31-40
pred: 11-20 | true: 21-30
pred: 21-30 | true: 11-20
pred: 11-20 | true: 21-30
pred: 11-20 | true: 11-20
pred: 11-20 | true: 11-20
pred: 21-30 | true: 11-20
pred: 21-30 | true: 11-20
pred: 21-30 | true: 21-30
pred: 21-30 | true: 31-40
pred: 21-30 | true: 11-20
pred: 21-30 | true: 21-30

12/30 correct
(This equals roughly 40.0% accur

### **Problems of the model**
-TBD-